The two-pointer technique uses two indices moving through a data structure to avoid the nested-loop brute force, typically trading O(n²) for O(n). Here are the key things to remember.

**The precondition is usually order.** Most two-pointer solutions require the data to be sorted (or have some monotonic property). The whole reason a pointer can move "confidently" in one direction is that order guarantees what lies beyond it. If the input isn't sorted and order matters, sorting first (O(n log n)) is often still worth it. Your triangle_number is exactly this: the sort is what licenses counting right - left in bulk.

**There are two main variants.** **Opposite-ends** (pointers start at both ends and converge — used for sorted-sum, container-with-most-water, palindrome checks) and **same-direction** (both start at the front, one races ahead — used for sliding windows, removing duplicates in place, fast/slow cycle detection). Identify which you need before writing code; they have different invariants.

**The core skill is the invariant: at each step, know why moving a pointer cannot skip a valid answer.** In opposite-ends sum problems, if a + b is too large you move the right pointer in because every element left of it is smaller and would only make the sum smaller still — so no answer is lost. Being able to state that sentence is the difference between a correct solution and one that silently misses cases. If you can't articulate why a move is safe, the algorithm is probably wrong.

**Each pointer should move monotonically.** The efficiency comes from each pointer traversing the array at most once, giving O(n). If you find yourself resetting a pointer backward inside the loop, you've likely collapsed back into O(n²) and should reconsider.

**Mind the boundary conditions.** The loop guard (left < right vs left <= right), where pointers start, and when you stop are the usual bug sources. Off-by-one errors here are extremely common — your loop's range(..., 1, -1) stopping point and the left < right guard are precisely the spots that need a comment, which is why you annotated them.

**Counting vs. collecting.** A subtle but powerful move: when order is established, a single found match can imply a whole batch (as in count += right - left). Watch for opportunities to count in bulk rather than enumerate one at a time — it's often what separates an accepted solution from a time-limit-exceeded one.

A quick mental test before committing: can I name the invariant, confirm each pointer moves one way only, and verify the boundary? If all three hold, the solution is usually sound.

In [10]:
from typing import List

def two_sum(sorted_nums: List[int], target: int) -> List[int] | None:
    """Find indices of two values in a sorted array that sum to a target.

    Assumes ``arr`` is sorted in ascending order. Uses two pointers from
    both ends: if the current sum is too small, advance the left pointer to
    increase it; if too large, retreat the right pointer to decrease it.
    This works precisely because the array is sorted.

    Args:
        sorted_nums (List[int]): Ascending-sorted sequence of numbers.
        target (int): Sum to search for.

    Returns: ``[left, right]`` indices of the matching pair, or ``None`` if
        no pair sums to the target.

    Example:
        >>> two_sum([1, 2, 3, 4, 5], 7)
        [1, 4]

    Note:
        See: https://www.hellointerview.com/learn/code/two-pointers/two-sum
    """
    left, right = 0, len(sorted_nums) - 1

    while left < right:
        tmp_sum = sorted_nums[left] + sorted_nums[right]
        if tmp_sum == target:
            return [left, right]
        elif tmp_sum < target:
            left += 1
        else:
            right -= 1

    return None

assert two_sum([1, 2, 3, 4, 5], 7) == [1,4]

In [7]:
from typing import List

def container_with_most_water(heights: List[int]) -> int:
    """Find the maximum water area between two vertical lines.

    Given an array where each element is the height of a vertical line at
    that index, find two lines that, together with the x-axis, form a
    container holding the most water. Area is bounded by the shorter line,
    so area = (right - left) * min(heights[left], heights[right]).

    Uses the two-pointer technique: start at both ends and move the pointer
    at the shorter line inward, since the shorter line is the binding
    constraint and moving the taller one can never increase the area.

    Args:
        heights (List[int]): Heights of the vertical lines, indexed by position.

    Returns:
        int: Maximum area of water the container can hold.

    Example:
        >>> container_with_most_water([3, 4, 1, 2, 2, 4, 1, 3, 2])
        21

    Note:
        See: https://www.hellointerview.com/learn/code/two-pointers/container-with-most-water
    """
    left, right, max_area = 0, len(heights) - 1, 0
    while left < right:
        width = right - left
        height = min(heights[left], heights[right])
        area = width * height

        max_area = max(max_area, area)

        if heights[left] < heights[right]:
            left += 1
        else:
            right -= 1

    return max_area

assert container_with_most_water([3, 4, 1, 2, 2, 4, 1, 3, 2]) == 21

In [8]:
from typing import List

def three_sum(nums: List[int]) -> List[List[int]]:
    """Find all unique triplets in ``nums`` that sum to zero.

    Sorts the input, then fixes each element ``nums[i]`` and runs a
    two-pointer search over the remainder for a pair summing to
    ``-nums[i]``. Sorting enables both the two-pointer scan and the
    duplicate-skipping that keeps the output free of repeated triplets.

    Note: sorts ``nums`` in place, so the caller's list is mutated.

    Time complexity: O(n^2). Sorting is O(n log n); the outer loop runs n
    times and each iteration's two-pointer scan is O(n), so the nested work
    dominates.

    Space complexity: O(n^2) counting the output, since there can be up to
    O(n^2) distinct triplets (each pair of values fixes the third).
    Auxiliary space excluding the output is O(1) beyond the sort.

    Args:
        nums (List[int]): Integers to search; order is not required (sorted internally).

    Returns:
        List[List[int]]: List of ``[a, b, c]`` triplets with ``a + b + c == 0``, each
        distinct by value, with ``a <= b <= c``.

    Example:
        >>> three_sum([-1, 0, 1, 2, -1, -4])
        [[-1, -1, 2], [-1, 0, 1]]

    Note:
        See: https://www.hellointerview.com/learn/code/two-pointers/3-sum
    """

    nums.sort()
    result = []

    # Fix the smallest element of each triplet. Stop at len - 2 so there is
    # always room for the left/right pair to its right.
    for i in range(len(nums) - 2):
        # Skip duplicate values of i: an identical nums[i] would only
        # reproduce triplets already found on the previous iteration.
        if i > 0 and nums[i] == nums[i - 1]:
            continue

        # Two-pointer search over the sub-array to the right of i, looking
        # for a pair that sums to -nums[i] (i.e. makes the total zero).
        # Reset the window to (i+1, end) fresh for each new i.
        left = i + 1
        right = len(nums) - 1

        while left < right:
            total = nums[i] + nums[left] + nums[right]

            if total < 0:
                left += 1        # Sum too small: raise it by moving left up.
            elif total > 0:
                right -= 1       # Sum too large: lower it by moving right down.
            else:
                result.append([nums[i], nums[left], nums[right]])

                # Advance both pointers past any duplicates so the next
                # iteration produces a distinct triplet, not a repeat of
                # this one. (This dedup is for left/right; the i-level
                # dedup above is handled separately.)
                while left < right and nums[left] == nums[left + 1]:
                    left += 1
                while left < right and nums[right] == nums[right - 1]:
                    right -= 1

                # Step onto the next pair of candidate values.
                left += 1
                right -= 1

    return result

assert three_sum([-1, 0, 1, 2, -1, -1]) == [[-1, -1, 2], [-1, 0, 1]]

In [9]:
from typing import List

def triangle_number(nums: List[int]) -> int:
    """Count the number of triplets in nums that can form a valid triangle.

    A triplet (a, b, c) forms a valid triangle if the sum of the two
    smaller sides is greater than the largest side. After sorting, for
    any fixed largest side nums[i], we use two pointers to count all
    valid pairs nums[left] and nums[right] where:

        nums[left] + nums[right] > nums[i]

    Args:
        nums (List[int]): A list of non-negative integers representing side lengths.

    Returns:
        The number of index triplets that can form a valid triangle.

    Example:
        >>> triangle_number([11,4,9,6,15,18])
        10

    Note:
        See: https://www.hellointerview.com/learn/code/two-pointers/valid-triangle-number
    """

    # Sorting lets us treat nums[i] as the largest side and use two pointers.
    nums.sort()

    count = 0

    # Fix nums[i] as the largest side. We need at least two elements before i.
    for i in range(len(nums) - 1, 1, -1):
        left = 0
        right = i - 1

        while left < right:
            # Since nums is sorted, if nums[left] + nums[right] > nums[i],
            # then every value from left through right - 1 also works with nums[right].
            if nums[left] + nums[right] > nums[i]:
                count += right - left
                right -= 1
            else:
                # Sum is too small, so move left forward to increase it.
                left += 1

    return count

assert triangle_number([11,4,9,6,15,18]) == 10

In [13]:
from typing import List


def swap(nums: List[int], i: int, j: int) -> None:
    """Swap the elements at indices i and j in place.

    Args:
        nums: The list to modify. Mutated in place.
        i: Index of the first element.
        j: Index of the second element.

    Note:
        Written in three-line tmp form rather than the idiomatic
        ``nums[i], nums[j] = nums[j], nums[i]``. The tuple form is
        correct and needs no temp (Python evaluates the whole right
        side before assigning), but the explicit form keeps the swap
        mechanically identical to its Java equivalent, which has no
        one-line tuple swap.
    """
    tmp = nums[j]
    nums[j] = nums[i]
    nums[i] = tmp


def move_zeroes(nums: List[int]) -> None:
    """Move all zeros to the end, preserving non-zero order, in place.

    Uses two pointers. The lead pointer (``scanner``) looks ahead for
    the next non-zero value; the trailing pointer (``next_non_zero``)
    marks the slot the next non-zero element should occupy. The gap
    between them equals the number of zeros seen so far.

    Args:
        nums: The list to modify. Mutated in place; no copy is made.

    Note:
        When the input has no zeros, or leading non-zero elements,
        each such element is swapped with itself (a no-op, since
        ``scanner == next_non_zero`` until the first zero is seen).
        Guarding with ``if scanner != next_non_zero`` would avoid the
        redundant swap, but trades a cheap swap for a cheap comparison
        and isn't worth it unless writes are genuinely expensive.

        Reference:
        https://www.hellointerview.com/learn/code/two-pointers/move-zeroes
    """
    next_non_zero = 0

    for scanner in range(len(nums)):
        if nums[scanner] != 0:
            swap(nums, next_non_zero, scanner)
            next_non_zero += 1


example = [2, 0, 4, 0, 9]
move_zeroes(example)
assert example == [2, 4, 9, 0, 0]